# Ejercicio 1.

Desarrolle un método para generar una variable aleatoria cuya densidad de probabilidad es

**a)**
$$
f(x) = \left\{
\begin{array}{ll}
\displaystyle\frac{x-2}{2}   & , 2 \leq x \leq 3\\
\displaystyle\frac{2-x/3}{2} & , 3 \leq x \leq 6\\
0 & , o.c.
\end{array}
\right. \,.
$$




Hacemos los cálculos en la pizzarra y vemos que:

$$
F(x) = \left\{
\begin{array}{ll}
0 & ,  x < 2 \\
\displaystyle\frac{(x-2)^2}{4} & ,  2 \le x \le 3 \\
\displaystyle x - \frac{x^2}{12} - 2 & ,  3 < x \le 6 \\
1 & ,  x > 6
\end{array}
\right.
$$

In [1]:
from random import random
import math

def simular_ej1a():
  U = random()
  if U < 0.25:
    return 2 + 2 * math.sqrt(U)
  else:
    return 6 - math.sqrt(12 * (1 - U))

Usamos la esperanza para revisar:

$$ E[X] = \int_{-\infty}^{\infty} x \cdot f(x) \, dx = \int_{2}^{3} x \left( \frac{x-2}{2} \right) dx + \int_{3}^{6} x \left( \frac{2 - x/3}{2} \right) dx
= \frac{1}{2} \left[ \frac{x^3}{3} - x^2 \right]_{2}^{3} + \frac{1}{2} \left[ x^2 - \frac{x^3}{9} \right]_{3}^{6} = \frac{5}{12} + \frac{39}{12} = \frac{11}{3} $$

In [2]:
n = 10000
esp = 0
for _ in range(n):
  esp += simular_ej1a()
esp /= n

print(f"Media estimada: {esp:.4f}")
print(f"Media teórica:  {11/3:.4f}")

Media estimada: 3.6714
Media teórica:  3.6667


# Ejercicio 3. Método de la composición:


**a)**
Suponga que es relativamente fácil generar $n$ variables aleatorias a partir
de sus distribuciones de probabilidad $F_i, \quad i=1,\dots,n$.

Implemente un método para generar una variable aleatoria cuya distribución de probabilidad es
$$
F(x) = \sum_{i=1}^n p_i \,F_i(x) \,.
$$
donde $p_i, \  i=1,\dots,n$, son números no negativos cuya suma es 1.

**b)** Genere datos usando tres exponenciales independientes con media 3, 5 y 7 respectivamente  y $p=(0.5,0.3,0.2)$. Calcule la esperanza exacta de la mezcla y estime con 10.000 repeticiones. Tenga cuidado con la parametrización que este usando!!

In [3]:
from random import random
import math
import numpy as np

def exponenciales_ej3():
    # probs = [0.5, 0.3, 0.2]
    # medias = [3, 5, 7]
    # 1. Generar U1 para seleccionar el tramo
    U1 = random()

    if U1 < 0.5:
        m = 3
    elif U1 < 0.8: #0.5 + 0.3
        m = 5
    else:
        m = 7

    # 2. Generar X a partir de la exponencial Fi elegida
    # Usamos el método de la transformada inversa: X = -ln(U)/lambda = -media * ln(U)
    U2 = 1-random()
    return (-m * math.log(U2))

def simular_composicion(nsim):
    muestras = []
    for _ in range(nsim):
        muestras.append(exponenciales_ej3())
    return muestras

In [4]:
# Ejecución y validación
n_reps = 10000
datos = simular_composicion(n_reps)
esperanza_estimada = np.mean(datos)
esperanza_teorica = 4.4

print(f"Resultados con {n_reps} repeticiones:")
print(f"Esperanza Estimada: {esperanza_estimada:.4f}")
print(f"Esperanza Teórica: {esperanza_teorica:.4f}")

Resultados con 10000 repeticiones:
Esperanza Estimada: 4.4007
Esperanza Teórica: 4.4000


# Ejercicio 7

**a)** Desarrolle dos métodos para generar una variable aleatoria $X$
con densidad de probabilidad:

$$\begin{cases}
\frac{1}{x} & , 1 \le x \le e\\
0 & ,o.c.
\end{cases}
$$


- i) Aplicando Transformada inversa.
- ii) Aplicando el método de aceptación y rechazo con una variable uniforme.

**b)** Compare la eficiencia de ambos métodos realizando $10.000$ simulaciones y comparando el promedio de los valores obtenidos. Compruebe que se obtiene un valor aproximado del valor esperado de $X$.

**c)**  Estime la probabilidad $P(X\le 2)$ y compárela con el valor real.

In [5]:
import random
import time
import numpy as np

**Inciso a)**

Cálculos en papel (pizarrón). Tenemos los siguientes métodos:

In [7]:
def Tinversa():
    u = random.random()
    return np.exp(u)

def Aceptacion_Rechazo():
  while 1:
    y = random.random() * (np.e-1) + 1     # Simulo Y (uniforme en (1, e))
    u = random.random()
    if u < 1/y:                          # Cota = f(y)/(c*g(y)) = (1/y) / ((e-1)*(1/e-1)) = (1/y)
      return y

**Inciso b)**

Notemos que $$
E(x)= \int_{1}^e x \frac{1}{x} dx = \int_{1}^e  1 dx = e-1 = 1.7183.
$$

In [8]:
def simulacion_TI_ejb(Nsim):
    start = time.time()
    sim = 0
    for _ in range(Nsim):
        sim = sim + Tinversa()
    sim = sim / Nsim
    end = time.time()
    return sim, end-start


def simulacion_AyR_ejb(Nsim):
    start = time.time()
    sim = 0
    for _ in range(Nsim):
        sim = sim + Aceptacion_Rechazo()
    sim = sim / Nsim
    end = time.time()
    return sim, end-start

In [9]:
Nsim = 10000

media, tiempo = simulacion_TI_ejb(Nsim)
print('Simulación TI: ', media)
print('Tiempo de simulación TI: ', tiempo)

media, tiempo = simulacion_AyR_ejb(Nsim)
print('\nSimulación AyR: ', media)
print('Tiempo de simulación AyR: ', tiempo)

Simulación TI:  1.7232913671321086
Tiempo de simulación TI:  0.011089324951171875

Simulación AyR:  1.7159826008116525
Tiempo de simulación AyR:  0.004926443099975586


**Inciso c)**

Notemos que:
$$
P(X\leq 2) = \int_{1}^{2} \frac{1}{x} dx = ln(2) - ln(1)= 0.6931.
$$

In [10]:
def simulacion_TI_ejc(Nsim):
    sim = 0
    for k in range(Nsim):
        x = Tinversa()
        if x <= 2:
            sim = sim + 1
    sim = sim / Nsim
    return sim

def simulacion_AyR_ejc(Nsim):
    sim = 0
    for k in range(Nsim):
        x = Aceptacion_Rechazo()
        if x <= 2:
            sim = sim + 1
    sim = sim / Nsim
    return sim

In [11]:
Nsim = 10000

print('Simulación probabilidad TI: ', simulacion_TI_ejc(Nsim))
print('Simulación probabilidad AyR: ', simulacion_AyR_ejc(Nsim))

Simulación probabilidad TI:  0.6979
Simulación probabilidad AyR:  0.6966
